# Recuperación y Reranking en Producción: El Pipeline Completo

En este notebook implementamos un pipeline de recuperación y reranking de producción. Iremos más allá del tutorial básico (TF-IDF + cosine) para mostrar los trade-offs reales y cómo construir algo que escale.

**Lo que veremos:**
1. Recuperación sparse: TF-IDF (baseline) y BM25 (producción)
2. Recuperación densa con embeddings + FAISS para ANN search
3. Búsqueda híbrida con Reciprocal Rank Fusion (RRF)
4. Reranking de segunda etapa con Cross-Encoder
5. Pipeline completo con evaluación y benchmark de latencia

**Prereqs:** `pip install scikit-learn sentence-transformers faiss-cpu`

## Arquitectura del Pipeline

Un pipeline de recuperación de producción tiene dos etapas principales:

```
Consulta
   │
   ▼
┌─────────────────────────────────────────┐
│  Etapa 1: Retrieval (rápido, alto recall)│
│  ┌──────────┐   ┌──────────┐            │
│  │  BM25    │   │  Dense   │  ──► RRF   │
│  │ (sparse) │   │(embeddings│            │
│  └──────────┘   └──────────┘            │
│                   top-20 candidatos      │
└─────────────────────┬───────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────┐
│  Etapa 2: Reranking (lento, alta        │
│           precisión)                    │
│           Cross-Encoder                 │
│           top-5 resultados finales      │
└─────────────────────┬───────────────────┘
                      │
                      ▼
                 LLM + contexto
```

**¿Por qué dos etapas?**
- La etapa 1 es rápida (O(log N) con FAISS) pero sacrifica algo de precisión → **recall alto**
- La etapa 2 es lenta (O(K) con K candidatos) pero muy precisa → **precision alta**
- Evaluamos el cross-encoder solo sobre ~20 candidatos, no sobre todo el corpus

In [12]:
import math
import time
import logging
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print('Imports OK')

Imports OK


In [13]:
@dataclass
class RetrievalConfig:
    """Configuración centralizada del pipeline de recuperación."""
    # BM25
    bm25_k1: float = 1.5    # Saturación de frecuencia de términos [1.2, 2.0]
    bm25_b: float = 0.75    # Normalización por longitud [0=off, 1=full]

    # Dense retrieval
    embedding_model: str = 'all-MiniLM-L6-v2'
    embedding_batch_size: int = 32
    use_faiss: bool = True

    # Cross-encoder reranking
    reranker_model: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
    reranker_batch_size: int = 16

    # Pipeline
    top_k_retrieval: int = 20   # Candidatos de primera etapa
    top_k_rerank: int = 5       # Resultados finales tras reranking
    rrf_k: int = 60             # Constante RRF (default empíricamente robusto)


config = RetrievalConfig()
print(config)

RetrievalConfig(bm25_k1=1.5, bm25_b=0.75, embedding_model='all-MiniLM-L6-v2', embedding_batch_size=32, use_faiss=True, reranker_model='cross-encoder/ms-marco-MiniLM-L-6-v2', reranker_batch_size=16, top_k_retrieval=20, top_k_rerank=5, rrf_k=60)


## 1. Corpus de Ejemplo

Para demostrar el pipeline usamos un corpus mixto:
- Pasajes narrativos del Quijote (en español, léxico arcaico)
- Conceptos técnicos sobre sistemas RAG (en español, léxico técnico)

Este contraste semántico permite ver cómo cada etapa filtra documentos relevantes de irrelevantes y dónde falla cada método.

In [14]:
CORPUS: List[str] = [
    # ── Quijote ───────────────────────────────────────────────────────────────
    "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor.",
    "El hidalgo tenía una olla de algo más vaca que carnero, salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes, algún palomino de añadidura los domingos.",
    "Era el hidalgo de los que suelen decir que no faltan a su obligación los caballeros; tenía en su casa una ama que pasaba de los cuarenta y una sobrina que no llegaba a los veinte.",
    "Frisaba la edad de nuestro hidalgo con los cincuenta años; era de complexión recia, seco de carnes, enjuto de rostro, gran madrugador y amigo de la caza.",
    "Con estas razones perdía el pobre caballero el juicio, y desvelábase por entenderlas y desentrañarles el sentido, que no se lo sacara ni las entendiera el mesmo Aristóteles.",
    "El cual Sancho Panza era un labrador vecino suyo, hombre de bien, pero de muy poca sal en la mollera.",
    "Don Quijote le propuso que se fuese con él de buena gana, porque tal vez le podría suceder aventura que ganase alguna ínsula y lo dejase a él por gobernador della.",
    "La Dulcinea del Toboso era una moza labradora de muy buen parecer, de quien él un tiempo anduvo enamorado, aunque, según se entiende, ella jamás lo supo ni se dio cata de ello.",
    "Los molinos de viento que en aquella llanura había, como serían hasta cuarenta, los tomó don Quijote por descomunales gigantes.",
    "Quiso la suerte que encontrase don Quijote a unos mercaderes toledanos que iban a comprar seda a Murcia, los cuales iban con sus quitasoles.",
    # ── RAG técnico ───────────────────────────────────────────────────────────
    "Los sistemas RAG (Retrieval-Augmented Generation) combinan un módulo de recuperación de información con un modelo generativo de lenguaje para producir respuestas precisas y actualizadas.",
    "La recuperación densa utiliza modelos de embedding para representar documentos y consultas en un espacio vectorial de alta dimensión, permitiendo búsqueda por similitud semántica.",
    "El reranking es la segunda etapa del pipeline: dado un conjunto inicial de candidatos recuperados, un modelo cross-encoder evalúa la relevancia de cada par consulta-documento de forma conjunta.",
    "BM25 es un algoritmo de recuperación probabilístico que mejora TF-IDF considerando la saturación de la frecuencia de términos (k1) y la normalización por longitud del documento (b).",
    "La búsqueda híbrida combina recuperación sparse (BM25) y densa (embeddings) usando Reciprocal Rank Fusion (RRF) para obtener lo mejor de ambos enfoques sin necesidad de normalizar scores.",
    "FAISS (Facebook AI Similarity Search) es una librería para búsqueda eficiente de similitud en grandes colecciones de vectores, con soporte para índices IVF, HNSW y cuantización por producto.",
    "Los cross-encoders toman el par consulta-documento concatenados y producen directamente una puntuación de relevancia; son más precisos que los bi-encoders pero más lentos en inferencia.",
    "En producción, el pipeline RAG incluye las etapas: chunking → embedding → indexación → recuperación → reranking → generación con LLM.",
    "El recall mide si encontramos los documentos relevantes; la precisión mide si los devueltos son realmente útiles. El reranking mejora la precisión sin sacrificar el recall inicial.",
    "Para RAG a gran escala, separar la indexación offline del retrieval online es crucial: la indexación puede tomar horas, pero el retrieval debe responder en menos de 100ms.",
]

# Ground truth: índices de documentos relevantes por consulta
GROUND_TRUTH: Dict[str, List[int]] = {
    "¿Dónde vive el hidalgo de la Mancha?": [0, 1, 2, 3],
    "¿Qué come don Quijote durante la semana?": [1],
    "¿Cómo funciona el reranking en sistemas RAG?": [12, 16, 18],
    "¿Qué ventajas tiene BM25 sobre TF-IDF?": [13],
    "¿Qué es FAISS y para qué se usa?": [15],
    "¿Quién es Sancho Panza?": [5, 6],
}

print(f'Corpus: {len(CORPUS)} documentos')
print(f'Consultas de evaluación: {len(GROUND_TRUTH)}')
for q in GROUND_TRUTH:
    print(f'  · {q}')

Corpus: 20 documentos
Consultas de evaluación: 6
  · ¿Dónde vive el hidalgo de la Mancha?
  · ¿Qué come don Quijote durante la semana?
  · ¿Cómo funciona el reranking en sistemas RAG?
  · ¿Qué ventajas tiene BM25 sobre TF-IDF?
  · ¿Qué es FAISS y para qué se usa?
  · ¿Quién es Sancho Panza?


## 2. Recuperación Sparse: TF-IDF y BM25

La recuperación sparse es la primera capa: rápida, sin GPU, y muy eficaz para coincidencias exactas de palabras clave.

### TF-IDF vs BM25

| Criterio | TF-IDF | BM25 |
|---|---|---|
| Saturación de frecuencia | No | Sí (parámetro k1) |
| Normalización por longitud | Parcial (L2) | Completa (parámetro b) |
| Resultado en benchmarks | Baseline | +5-15% sobre TF-IDF |
| Implementación disponible | `sklearn` | `rank-bm25` o manual |

**Regla de producción**: usar siempre BM25. TF-IDF solo como referencia para medir el delta.

In [15]:
class TFIDFRetriever:
    """Recuperador sparse TF-IDF — sirve como baseline de referencia."""

    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            analyzer='word',
            ngram_range=(1, 2),   # uni + bigramas
            sublinear_tf=True,    # log(1 + tf) en lugar de tf crudo
            min_df=1,
        )
        self.doc_vectors = None
        self.corpus: List[str] = []

    def build_index(self, corpus: List[str]) -> None:
        self.corpus = corpus
        self.doc_vectors = self.vectorizer.fit_transform(corpus)
        n_features = self.doc_vectors.shape[1]
        logger.info(f'TF-IDF index: {len(corpus)} docs, {n_features} features')

    def retrieve(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.doc_vectors)[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in top_indices if scores[i] > 0]


# ── Demo ──────────────────────────────────────────────────────────────────────
tfidf = TFIDFRetriever()
tfidf.build_index(CORPUS)

query = '¿Dónde vive el hidalgo de la Mancha?'
results = tfidf.retrieve(query, top_k=5)

print(f'Query: "{query}"\n')
print('Top 5 resultados TF-IDF:')
for rank, (idx, score) in enumerate(results, 1):
    relevante = '✓' if idx in GROUND_TRUTH[query] else ' '
    print(f'  [{relevante}] {rank}. [{score:.4f}] {CORPUS[idx][:75]}...')

INFO: TF-IDF index: 20 docs, 788 features


Query: "¿Dónde vive el hidalgo de la Mancha?"

Top 5 resultados TF-IDF:
  [✓] 1. [0.2647] En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho t...
  [✓] 2. [0.1492] Era el hidalgo de los que suelen decir que no faltan a su obligación los ca...
  [✓] 3. [0.1239] Frisaba la edad de nuestro hidalgo con los cincuenta años; era de complexió...
  [✓] 4. [0.1116] El hidalgo tenía una olla de algo más vaca que carnero, salpicón las más no...
  [ ] 5. [0.0913] BM25 es un algoritmo de recuperación probabilístico que mejora TF-IDF consi...


### 2.2 BM25: El Estándar de Producción para Recuperación Sparse

BM25 añade dos mejoras sobre TF-IDF:

**Saturación de TF** (parámetro `k1`): un término que aparece 100 veces no debe valer 100x más que uno que aparece una vez. La curva de saturación limita el peso:

$$\text{score}_{\text{BM25}} = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

**Normalización por longitud** (parámetro `b`): documentos largos no deben tener ventaja injusta por acumular más términos.

- `k1 ∈ [1.2, 2.0]`: valores altos = menos saturación (más peso a alta frecuencia)
- `b ∈ [0, 1]`: 0 = sin normalización de longitud, 1 = normalización completa

In [16]:
class BM25Retriever:
    """
    Recuperador BM25 implementado sobre numpy.

    En producción se recomienda `rank-bm25` por velocidad, pero esta
    implementación muestra el algoritmo de forma explícita y evita
    dependencias adicionales.
    """

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus: List[str] = []
        self.doc_freqs: List[Dict[str, int]] = []
        self.idf: Dict[str, float] = {}
        self.avgdl: float = 0.0
        self.doc_lengths: List[int] = []

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return text.lower().split()

    def build_index(self, corpus: List[str]) -> None:
        self.corpus = corpus
        self.doc_freqs = []
        df: Dict[str, int] = defaultdict(int)

        for doc in corpus:
            tokens = self._tokenize(doc)
            freq: Dict[str, int] = defaultdict(int)
            for token in tokens:
                freq[token] += 1
            self.doc_freqs.append(dict(freq))
            self.doc_lengths.append(len(tokens))
            for token in set(tokens):
                df[token] += 1

        N = len(corpus)
        self.avgdl = float(np.mean(self.doc_lengths))
        # IDF con suavizado de Robertson (evita negativos)
        self.idf = {
            term: math.log((N - cnt + 0.5) / (cnt + 0.5) + 1)
            for term, cnt in df.items()
        }
        logger.info(
            f'BM25 index: {N} docs, vocab={len(self.idf)}, avgdl={self.avgdl:.1f}'
        )

    def _score_doc(self, query_terms: List[str], doc_idx: int) -> float:
        dl = self.doc_lengths[doc_idx]
        freq = self.doc_freqs[doc_idx]
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue
            tf = freq.get(term, 0)
            num = tf * (self.k1 + 1)
            den = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += self.idf[term] * num / den
        return score

    def retrieve(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        query_terms = self._tokenize(query)
        scores = [self._score_doc(query_terms, i) for i in range(len(self.corpus))]
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in top_indices if scores[i] > 0]


# ── Demo y comparación con TF-IDF ─────────────────────────────────────────────
bm25 = BM25Retriever(k1=config.bm25_k1, b=config.bm25_b)
bm25.build_index(CORPUS)

query = '¿Dónde vive el hidalgo de la Mancha?'
r_tfidf = tfidf.retrieve(query, top_k=5)
r_bm25 = bm25.retrieve(query, top_k=5)

print(f'Query: "{query}"\n')
print(f'{"TF-IDF":<52} {"BM25":<52}')
print('-' * 106)
for (i1, s1), (i2, s2) in zip(r_tfidf, r_bm25):
    t1 = f'[{s1:.4f}] {CORPUS[i1][:44]}'
    t2 = f'[{s2:.4f}] {CORPUS[i2][:44]}'
    print(f'{t1:<52} {t2:<52}')

INFO: BM25 index: 20 docs, vocab=315, avgdl=27.0


Query: "¿Dónde vive el hidalgo de la Mancha?"

TF-IDF                                               BM25                                                
----------------------------------------------------------------------------------------------------------
[0.2647] En un lugar de la Mancha, de cuyo nombre no  [2.8645] Frisaba la edad de nuestro hidalgo con los c
[0.1492] Era el hidalgo de los que suelen decir que n [2.5949] El hidalgo tenía una olla de algo más vaca q
[0.1239] Frisaba la edad de nuestro hidalgo con los c [2.3672] En un lugar de la Mancha, de cuyo nombre no 
[0.1116] El hidalgo tenía una olla de algo más vaca q [2.3405] Era el hidalgo de los que suelen decir que n
[0.0913] BM25 es un algoritmo de recuperación probabi [2.1823] El recall mide si encontramos los documentos


## 3. Recuperación Densa con Embeddings

La recuperación densa captura similitud **semántica** que la sparse no puede: sinónimos, paráfrasis y relaciones conceptuales. El bi-encoder genera un vector para el documento *una sola vez* (offline) y otro para la consulta (online), luego calcula la similitud.

### Búsqueda exacta vs aproximada (ANN)

| Método | Complejidad | Uso |
|---|---|---|
| Exacto (`cosine_similarity`) | O(N) | Corpus < 10K docs |
| FAISS IVF (ANN) | O(√N) aprox. | Corpus > 10K docs |
| FAISS HNSW (ANN) | O(log N) aprox. | Alta frecuencia de queries |

**Regla de producción**: siempre usar ANN para corpus > 10K documentos. La pérdida de recall es marginal (<1%) y la ganancia de latencia es de 10-100x.

In [17]:
class DenseRetriever:
    """Recuperador denso: bi-encoder + FAISS para búsqueda aproximada (ANN)."""

    def __init__(self, model_name: str, batch_size: int = 32, use_faiss: bool = True):
        self.model_name = model_name
        self.batch_size = batch_size
        self.use_faiss = use_faiss
        self.model = None
        self.corpus: List[str] = []
        self.doc_embeddings: Optional[np.ndarray] = None
        self.index = None

    def _load_model(self) -> None:
        if self.model is None:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(self.model_name)
            logger.info(f'Modelo cargado: {self.model_name}')

    def build_index(self, corpus: List[str]) -> None:
        self._load_model()
        self.corpus = corpus

        logger.info(f'Generando embeddings para {len(corpus)} documentos...')
        self.doc_embeddings = self.model.encode(
            corpus,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,  # L2-norm → cosine = dot product
        )

        if self.use_faiss:
            self._build_faiss_index()

        dim = self.doc_embeddings.shape[1]
        logger.info(f'Dense index listo: {len(corpus)} docs, dim={dim}')

    def _build_faiss_index(self) -> None:
        try:
            import faiss
            dim = self.doc_embeddings.shape[1]
            n = len(self.corpus)

            if n < 256:
                # Corpus pequeño → búsqueda exacta (no hace falta ANN)
                self.index = faiss.IndexFlatIP(dim)
            else:
                # IVFFlat: buena relación velocidad/precisión para corpus medianos
                n_clusters = max(4, int(math.sqrt(n)))
                quantizer = faiss.IndexFlatIP(dim)
                self.index = faiss.IndexIVFFlat(
                    quantizer, dim, n_clusters, faiss.METRIC_INNER_PRODUCT
                )
                self.index.train(self.doc_embeddings.astype(np.float32))
                # nprobe controla el tradeoff velocidad/recall
                self.index.nprobe = max(1, n_clusters // 4)

            self.index.add(self.doc_embeddings.astype(np.float32))
            logger.info(f'FAISS {type(self.index).__name__}: {self.index.ntotal} vectores')

        except ImportError:
            logger.warning('FAISS no disponible — usando búsqueda exacta como fallback')
            self.use_faiss = False

    def retrieve(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        self._load_model()
        q_emb = self.model.encode(
            [query], normalize_embeddings=True
        )[0].reshape(1, -1).astype(np.float32)

        if self.use_faiss and self.index is not None:
            scores, indices = self.index.search(q_emb, top_k)
            return [
                (int(i), float(s))
                for i, s in zip(indices[0], scores[0])
                if i >= 0
            ]

        # Fallback: búsqueda exacta
        sims = cosine_similarity(q_emb, self.doc_embeddings)[0]
        top = np.argsort(sims)[::-1][:top_k]
        return [(int(i), float(sims[i])) for i in top]


# ── Demo ──────────────────────────────────────────────────────────────────────
dense = DenseRetriever(
    model_name=config.embedding_model,
    batch_size=config.embedding_batch_size,
    use_faiss=config.use_faiss,
)
dense.build_index(CORPUS)

query = '¿Dónde vive el hidalgo de la Mancha?'
results_dense = dense.retrieve(query, top_k=5)

print(f'\nQuery: "{query}"\n')
print('Top 5 resultados Dense (embeddings + FAISS):')
for rank, (idx, score) in enumerate(results_dense, 1):
    relevante = '✓' if idx in GROUND_TRUTH[query] else ' '
    print(f'  [{relevante}] {rank}. [{score:.4f}] {CORPUS[idx][:75]}...')

INFO: No device provided, using cuda:0
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO


Query: "¿Dónde vive el hidalgo de la Mancha?"

Top 5 resultados Dense (embeddings + FAISS):
  [✓] 1. [0.7246] En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho t...
  [✓] 2. [0.6794] El hidalgo tenía una olla de algo más vaca que carnero, salpicón las más no...
  [✓] 3. [0.6560] Era el hidalgo de los que suelen decir que no faltan a su obligación los ca...
  [✓] 4. [0.6289] Frisaba la edad de nuestro hidalgo con los cincuenta años; era de complexió...
  [ ] 5. [0.5381] Con estas razones perdía el pobre caballero el juicio, y desvelábase por en...


## 4. Búsqueda Híbrida con Reciprocal Rank Fusion (RRF)

BM25 y Dense son complementarios:
- **BM25** domina en coincidencias exactas de palabras clave y términos técnicos
- **Dense** domina en paráfrasis, sinónimos y comprensión semántica

**Reciprocal Rank Fusion** combina los rankings de forma robusta:

$$\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}$$

**¿Por qué RRF en lugar de suma ponderada de scores?**
- BM25 y cosine tienen escalas totalmente diferentes → normalizar es difícil
- RRF solo usa los rankings, no los scores absolutos → robusto a outliers
- El parámetro `k=60` funciona bien en casi todos los benchmarks sin tuning

In [18]:
def reciprocal_rank_fusion(
    rankings: List[List[Tuple[int, float]]],
    k: int = 60,
) -> List[Tuple[int, float]]:
    """Combina N rankings con RRF. Devuelve lista ordenada de (doc_id, rrf_score)."""
    rrf_scores: Dict[int, float] = defaultdict(float)
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking, 1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


class HybridRetriever:
    """Combina BM25 + Dense mediante Reciprocal Rank Fusion."""

    def __init__(self, sparse: BM25Retriever, dense: DenseRetriever, rrf_k: int = 60):
        self.sparse = sparse
        self.dense = dense
        self.rrf_k = rrf_k

    def retrieve(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        # Recuperar más candidatos de cada fuente para mejorar el recall del RRF
        sparse_results = self.sparse.retrieve(query, top_k=top_k * 2)
        dense_results = self.dense.retrieve(query, top_k=top_k * 2)
        combined = reciprocal_rank_fusion(
            [sparse_results, dense_results], k=self.rrf_k
        )
        return combined[:top_k]


# ── Demo comparativo ───────────────────────────────────────────────────────────
hybrid = HybridRetriever(bm25, dense, rrf_k=config.rrf_k)

print('Comparación de Recall@5 por método\n')
print(f'{"Query":<45} {"BM25":>6} {"Dense":>6} {"Hybrid":>7}')
print('-' * 70)

for query, relevant in GROUND_TRUTH.items():
    rel_set = set(relevant)
    r_bm25   = {idx for idx, _ in bm25.retrieve(query, top_k=5)}
    r_dense  = {idx for idx, _ in dense.retrieve(query, top_k=5)}
    r_hybrid = {idx for idx, _ in hybrid.retrieve(query, top_k=5)}

    recall = lambda r: len(r & rel_set) / max(len(rel_set), 1)
    print(
        f'{query[:44]:<45} '
        f'{recall(r_bm25):>6.2f} '
        f'{recall(r_dense):>6.2f} '
        f'{recall(r_hybrid):>7.2f}'
    )

Comparación de Recall@5 por método

Query                                           BM25  Dense  Hybrid
----------------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00, 102.89it/s]


¿Dónde vive el hidalgo de la Mancha?            1.00   1.00    1.00


Batches: 100%|██████████| 1/1 [00:00<00:00, 122.64it/s]


¿Qué come don Quijote durante la semana?        0.00   0.00    0.00


...

¿Qué ventajas tiene BM25 sobre TF-IDF?          1.00   1.00    1.00


Batches: 100%|██████████| 1/1 [00:00<00:00, 117.80it/s]


¿Qué es FAISS y para qué se usa?                1.00   0.00    1.00


Batches: 100%|██████████| 1/1 [00:00<00:00, 95.71it/s]

¿Quién es Sancho Panza?                         0.50   0.50    0.50


## 5. Reranking con Cross-Encoder

El bi-encoder genera embeddings independientes para consulta y documento → rápido pero pierde interacciones finas.

El **cross-encoder** procesa el par `[consulta, documento]` de forma conjunta → captura interacciones a nivel de token, mucho más preciso.

| Aspecto | Bi-encoder | Cross-encoder |
|---|---|---|
| Entrada | doc y query por separado | `[CLS] query [SEP] doc [SEP]` |
| Complejidad | O(1) por doc en tiempo real | O(K) sobre K candidatos |
| Latencia (20 docs) | ~1ms | ~20-100ms |
| Precisión | Buena | Muy alta |

**Patrón de producción**: bi-encoder recupera top-20, cross-encoder reordena a top-5.

In [19]:
class CrossEncoderReranker:
    """
    Reranker de segunda etapa usando cross-encoder.

    Nota: el modelo por defecto está entrenado en inglés (MS MARCO).
    Para RAG en español en producción, usar un modelo multilingüe como
    'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'.
    """

    def __init__(self, model_name: str, batch_size: int = 16):
        self.model_name = model_name
        self.batch_size = batch_size
        self.model = None

    def _load_model(self) -> None:
        if self.model is None:
            try:
                from sentence_transformers import CrossEncoder
                self.model = CrossEncoder(self.model_name)
                logger.info(f'Cross-encoder cargado: {self.model_name}')
            except Exception as e:
                logger.warning(f'Cross-encoder no disponible: {e}. Usando orden del retriever.')

    def rerank(
        self,
        query: str,
        candidates: List[Tuple[int, str]],  # [(doc_id, doc_text), ...]
        top_k: int = 5,
    ) -> List[Tuple[int, float]]:
        """Reranking de candidatos. Devuelve [(doc_id, score)] ordenado."""
        self._load_model()

        if not candidates:
            return []

        if self.model is None:
            # Fallback: devolver candidatos en el orden original del retriever
            return [(doc_id, -float(rank)) for rank, (doc_id, _) in enumerate(candidates[:top_k])]

        pairs = [[query, text] for _, text in candidates]
        scores = self.model.predict(pairs, batch_size=self.batch_size)

        ranked = sorted(
            zip([doc_id for doc_id, _ in candidates], scores),
            key=lambda x: x[1],
            reverse=True,
        )
        return ranked[:top_k]


# ── Demo ──────────────────────────────────────────────────────────────────────
reranker = CrossEncoderReranker(
    model_name=config.reranker_model,
    batch_size=config.reranker_batch_size,
)

query = '¿Cómo funciona el reranking en sistemas RAG?'
candidates_hybrid = hybrid.retrieve(query, top_k=config.top_k_retrieval)
candidates_with_text = [(idx, CORPUS[idx]) for idx, _ in candidates_hybrid]

reranked = reranker.rerank(query, candidates_with_text, top_k=config.top_k_rerank)

relevant = set(GROUND_TRUTH[query])
print(f'Query: "{query}"\n')
print(f'Antes del reranking — top 5 de {config.top_k_retrieval} candidatos híbridos:')
for rank, (idx, _) in enumerate(candidates_hybrid[:5], 1):
    r = '✓' if idx in relevant else ' '
    print(f'  [{r}] {rank}. {CORPUS[idx][:75]}...')

print(f'\nDespués del cross-encoder — top {config.top_k_rerank}:')
for rank, (idx, score) in enumerate(reranked, 1):
    r = '✓' if idx in relevant else ' '
    print(f'  [{r}] {rank}. [{score:.4f}] {CORPUS[idx][:75]}...')

Batches: 100%|██████████| 1/1 [00:00<00:00, 157.00it/s]
INFO: No device provided, using cuda:0
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
INFO: No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
INFO: HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporar

Query: "¿Cómo funciona el reranking en sistemas RAG?"

Antes del reranking — top 5 de 20 candidatos híbridos:
  [ ] 1. En producción, el pipeline RAG incluye las etapas: chunking → embedding → i...
  [ ] 2. Los sistemas RAG (Retrieval-Augmented Generation) combinan un módulo de rec...
  [✓] 3. El recall mide si encontramos los documentos relevantes; la precisión mide ...
  [✓] 4. El reranking es la segunda etapa del pipeline: dado un conjunto inicial de ...
  [ ] 5. Con estas razones perdía el pobre caballero el juicio, y desvelábase por en...

Después del cross-encoder — top 5:
  [ ] 1. [2.5736] Los sistemas RAG (Retrieval-Augmented Generation) combinan un módulo de rec...
  [ ] 2. [2.5028] En producción, el pipeline RAG incluye las etapas: chunking → embedding → i...
  [✓] 3. [0.2583] El reranking es la segunda etapa del pipeline: dado un conjunto inicial de ...
  [✓] 4. [-4.1846] El recall mide si encontramos los documentos relevantes; la precisión mide ...
  [ ] 5. [-5.5822] Para R

## 6. Pipeline Completo de Producción

Integramos todo en `RAGRetriever`, que gestiona dos fases bien separadas:

- **Fase offline** (`build_index`): indexar documentos → puede tardar minutos/horas
- **Fase online** (`retrieve`): responder queries → debe tomar <100ms

Esta separación es el patrón central de producción: el índice se construye una vez y se sirve indefinidamente hasta que los datos cambian.

In [20]:
@dataclass
class RetrievalResult:
    doc_id: int
    text: str
    score: float
    stage: str   # 'retrieval' | 'rerank'
    rank: int


class RAGRetriever:
    """Pipeline completo de recuperación y reranking."""

    def __init__(self, corpus: List[str], cfg: RetrievalConfig):
        self.corpus = corpus
        self.cfg = cfg
        self._indexed = False

        self.bm25 = BM25Retriever(k1=cfg.bm25_k1, b=cfg.bm25_b)
        self.dense = DenseRetriever(
            model_name=cfg.embedding_model,
            batch_size=cfg.embedding_batch_size,
            use_faiss=cfg.use_faiss,
        )
        self.hybrid = HybridRetriever(self.bm25, self.dense, rrf_k=cfg.rrf_k)
        self.reranker = CrossEncoderReranker(
            model_name=cfg.reranker_model,
            batch_size=cfg.reranker_batch_size,
        )

    def build_index(self) -> None:
        """Fase offline: construir todos los índices."""
        t0 = time.perf_counter()
        self.bm25.build_index(self.corpus)
        self.dense.build_index(self.corpus)
        self._indexed = True
        logger.info(f'Índice completo construido en {time.perf_counter() - t0:.2f}s')

    def retrieve(
        self,
        query: str,
        use_reranking: bool = True,
    ) -> List[RetrievalResult]:
        """Fase online: recuperar y rerankear."""
        if not self._indexed:
            raise RuntimeError('Llamar build_index() antes de retrieve()')

        t0 = time.perf_counter()

        # Etapa 1: recuperación híbrida
        candidates = self.hybrid.retrieve(query, top_k=self.cfg.top_k_retrieval)
        t1 = time.perf_counter()

        if use_reranking:
            # Etapa 2: reranking con cross-encoder
            candidates_with_text = [(idx, self.corpus[idx]) for idx, _ in candidates]
            reranked = self.reranker.rerank(
                query, candidates_with_text, top_k=self.cfg.top_k_rerank
            )
            results = [
                RetrievalResult(idx, self.corpus[idx], score, 'rerank', rank + 1)
                for rank, (idx, score) in enumerate(reranked)
            ]
        else:
            results = [
                RetrievalResult(idx, self.corpus[idx], score, 'retrieval', rank + 1)
                for rank, (idx, score) in enumerate(candidates[: self.cfg.top_k_rerank])
            ]

        t2 = time.perf_counter()
        logger.debug(
            f'retrieval={1000*(t1-t0):.1f}ms '
            f'rerank={1000*(t2-t1):.1f}ms '
            f'total={1000*(t2-t0):.1f}ms'
        )
        return results


# ── Construir índice y demo con todas las queries ─────────────────────────────
rag = RAGRetriever(CORPUS, config)
rag.build_index()

print('\n' + '=' * 80)
print('PIPELINE COMPLETO — DEMO')
print('=' * 80)

for query, relevant in GROUND_TRUTH.items():
    results = rag.retrieve(query)
    rel_set = set(relevant)
    hits = sum(1 for r in results if r.doc_id in rel_set)
    print(f'\nQuery: "{query}"  [hits={hits}/{len(relevant)}]')
    for r in results:
        mark = '✓' if r.doc_id in rel_set else ' '
        print(f'  [{mark}] {r.rank}. [{r.score:.4f}] {r.text[:72]}...')

INFO: BM25 index: 20 docs, vocab=315, avgdl=27.0
INFO: No device provided, using cuda:0
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transfo


PIPELINE COMPLETO — DEMO


Batches: 100%|██████████| 1/1 [00:00<00:00, 157.65it/s]
INFO: No device provided, using cuda:0
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
INFO: No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
INFO: HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporar


Query: "¿Dónde vive el hidalgo de la Mancha?"  [hits=4/4]
  [✓] 1. [5.5930] En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha much...
  [✓] 2. [0.1080] Era el hidalgo de los que suelen decir que no faltan a su obligación los...
  [✓] 3. [0.0145] El hidalgo tenía una olla de algo más vaca que carnero, salpicón las más...
  [✓] 4. [-1.9676] Frisaba la edad de nuestro hidalgo con los cincuenta años; era de comple...
  [ ] 5. [-7.7954] Don Quijote le propuso que se fuese con él de buena gana, porque tal vez...


Batches: 100%|██████████| 2/2 [00:00<00:00, 110.00it/s]


...


Query: "¿Qué ventajas tiene BM25 sobre TF-IDF?"  [hits=1/1]
  [✓] 1. [2.5178] BM25 es un algoritmo de recuperación probabilístico que mejora TF-IDF co...
  [ ] 2. [-5.0675] La búsqueda híbrida combina recuperación sparse (BM25) y densa (embeddin...
  [ ] 3. [-10.8275] En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha much...
  [ ] 4. [-10.8727] Era el hidalgo de los que suelen decir que no faltan a su obligación los...
  [ ] 5. [-11.0426] FAISS (Facebook AI Similarity Search) es una librería para búsqueda efic...


Batches: 100%|██████████| 2/2 [00:00<00:00, 94.60it/s]



Query: "¿Qué es FAISS y para qué se usa?"  [hits=1/1]
  [✓] 1. [0.9577] FAISS (Facebook AI Similarity Search) es una librería para búsqueda efic...
  [ ] 2. [-8.6071] Don Quijote le propuso que se fuese con él de buena gana, porque tal vez...
  [ ] 3. [-8.7741] Con estas razones perdía el pobre caballero el juicio, y desvelábase por...
  [ ] 4. [-9.2794] Quiso la suerte que encontrase don Quijote a unos mercaderes toledanos q...
  [ ] 5. [-9.6656] La Dulcinea del Toboso era una moza labradora de muy buen parecer, de qu...


Batches: 100%|██████████| 2/2 [00:00<00:00, 65.87it/s]


Query: "¿Quién es Sancho Panza?"  [hits=1/2]
  [✓] 1. [3.2880] El cual Sancho Panza era un labrador vecino suyo, hombre de bien, pero d...
  [ ] 2. [-6.8055] La Dulcinea del Toboso era una moza labradora de muy buen parecer, de qu...
  [ ] 3. [-7.4189] Quiso la suerte que encontrase don Quijote a unos mercaderes toledanos q...
  [ ] 4. [-8.1723] Frisaba la edad de nuestro hidalgo con los cincuenta años; era de comple...
  [ ] 5. [-8.4907] En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha much...


## 7. Evaluación del Pipeline

Las métricas de evaluación para retrieval:

| Métrica | Qué mide | Fórmula |
|---|---|---|
| **Recall@K** | ¿Encontramos los relevantes? | `|retrieved ∩ relevant| / |relevant|` |
| **Precision@K** | ¿Los devueltos son útiles? | `|retrieved ∩ relevant| / K` |
| **MRR** | Posición del primer relevante | `mean(1 / rank_of_first_relevant)` |

En el notebook `03_evaluation_metrics.ipynb` se cubren NDCG, LLM-as-judge y análisis de fallos en detalle. Aquí comparamos los métodos implementados.

In [10]:
def recall_at_k(retrieved: List[int], relevant: List[int], k: int) -> float:
    return len(set(retrieved[:k]) & set(relevant)) / max(len(relevant), 1)


def precision_at_k(retrieved: List[int], relevant: List[int], k: int) -> float:
    return sum(1 for i in retrieved[:k] if i in set(relevant)) / k


def mean_reciprocal_rank(retrieved: List[int], relevant: Set[int]) -> float:
    for rank, idx in enumerate(retrieved, 1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0


def evaluate_method(retrieve_fn, ground_truth: Dict[str, List[int]], k: int = 5):
    """Evalúa un método de retrieval sobre todas las queries del ground truth."""
    recalls, precisions, mrrs = [], [], []
    for query, relevant in ground_truth.items():
        retrieved = retrieve_fn(query)
        recalls.append(recall_at_k(retrieved, relevant, k))
        precisions.append(precision_at_k(retrieved, relevant, k))
        mrrs.append(mean_reciprocal_rank(retrieved, set(relevant)))
    return {
        f'Recall@{k}': np.mean(recalls),
        f'Precision@{k}': np.mean(precisions),
        'MRR': np.mean(mrrs),
    }


# ── Comparación ───────────────────────────────────────────────────────────────
K = 5
methods = {
    'TF-IDF':        lambda q: [i for i, _ in tfidf.retrieve(q, top_k=K)],
    'BM25':          lambda q: [i for i, _ in bm25.retrieve(q, top_k=K)],
    'Dense':         lambda q: [i for i, _ in dense.retrieve(q, top_k=K)],
    'Hybrid (RRF)':  lambda q: [i for i, _ in hybrid.retrieve(q, top_k=K)],
    'Hybrid+Rerank': lambda q: [r.doc_id for r in rag.retrieve(q)],
}

print(f'Evaluación sobre {len(GROUND_TRUTH)} queries (K={K})\n')
print(f'{"Método":<18} {f"Recall@{K}":>10} {f"Precision@{K}":>13} {"MRR":>8}')
print('-' * 55)

for name, fn in methods.items():
    metrics = evaluate_method(fn, GROUND_TRUTH, k=K)
    r = metrics[f'Recall@{K}']
    p = metrics[f'Precision@{K}']
    m = metrics['MRR']
    print(f'{name:<18} {r:>10.3f} {p:>13.3f} {m:>8.3f}')

Evaluación sobre 6 queries (K=5)

Método               Recall@5   Precision@5      MRR
-------------------------------------------------------
TF-IDF                  0.694         0.300    0.722
BM25                    0.694         0.300    0.750


Batches: 100%|██████████| 1/1 [00:00<00:00, 144.61it/s]


Dense                   0.528         0.267    0.556


Batches: 100%|██████████| 1/1 [00:00<00:00, 193.17it/s]


Hybrid (RRF)            0.694         0.300    0.597


Batches: 100%|██████████| 2/2 [00:00<00:00, 87.08it/s]

Hybrid+Rerank           0.694         0.300    0.722


## 8. Benchmark de Latencia

En producción la latencia importa tanto como la calidad. Desglose típico:

| Componente | Latencia | Escalabilidad |
|---|---|---|
| BM25 | ~1ms | O(|vocab| × |query terms|) |
| Dense (FAISS IVF) | ~5ms | O(√N) aproximado |
| Cross-encoder (20 docs) | ~50-150ms | O(K) lineal en candidatos |

El cross-encoder domina la latencia pero solo evalúa K=20 pares, no el corpus completo.

In [11]:
def benchmark(fn, queries: List[str], n_runs: int = 3) -> Dict[str, float]:
    """Mide latencia de una función de retrieval en ms."""
    times_ms = []
    # Primer run para calentar el modelo/caché
    for _ in range(n_runs + 1):
        for q in queries:
            t0 = time.perf_counter()
            fn(q)
            times_ms.append((time.perf_counter() - t0) * 1000)
    # Descartar el primer run (warm-up)
    times_ms = times_ms[len(queries):]
    return {
        'mean': np.mean(times_ms),
        'p50':  np.percentile(times_ms, 50),
        'p95':  np.percentile(times_ms, 95),
        'p99':  np.percentile(times_ms, 99),
    }


test_queries = list(GROUND_TRUTH.keys())

bench_fns = {
    'BM25':             lambda q: bm25.retrieve(q, top_k=5),
    'Dense (FAISS)':    lambda q: dense.retrieve(q, top_k=5),
    'Hybrid (RRF)':     lambda q: hybrid.retrieve(q, top_k=20),
    'Hybrid+Rerank':    lambda q: rag.retrieve(q),
}

print(f'Benchmark de latencia (ms) — {len(test_queries)} queries, 3 runs\n')
print(f'{"Método":<22} {"Mean":>7} {"P50":>7} {"P95":>7} {"P99":>7}')
print('-' * 56)

for name, fn in bench_fns.items():
    stats = benchmark(fn, test_queries)
    print(
        f'{name:<22} '
        f'{stats["mean"]:>6.1f}  '
        f'{stats["p50"]:>6.1f}  '
        f'{stats["p95"]:>6.1f}  '
        f'{stats["p99"]:>6.1f}'
    )

print('\nNota: corpus de 20 docs. Escalar el corpus reduce latencia relativa del BM25')
print('      y aumenta la ventaja de FAISS sobre búsqueda exacta.')

Benchmark de latencia (ms) — 6 queries, 3 runs

Método                    Mean     P50     P95     P99
--------------------------------------------------------
BM25                      0.0     0.0     0.1     0.1


Batches: 100%|██████████| 1/1 [00:00<00:00, 209.27it/s]


Dense (FAISS)            14.6    13.2    28.2    32.4


Batches: 100%|██████████| 1/1 [00:00<00:00, 157.50it/s]


Hybrid (RRF)             16.8    12.9    30.1    31.8


Batches: 100%|██████████| 2/2 [00:00<00:00, 57.95it/s]

Hybrid+Rerank            47.5    48.3    63.8    66.7

Nota: corpus de 20 docs. Escalar el corpus reduce latencia relativa del BM25
      y aumenta la ventaja de FAISS sobre búsqueda exacta.


## 9. Buenas Prácticas para Producción

### Lo que sí hacer

1. **Separar indexación y retrieval**: el índice se construye offline; el retrieval es online. No mezclarlos en el mismo proceso.
2. **Warm-up de modelos al arrancar**: cargar los pesos y hacer una query dummy antes de servir tráfico real.
3. **Cachear embeddings de queries frecuentes**: las queries se repiten. Un caché LRU de embeddings ahorra 80% del cómputo dense.
4. **Ajustar `top_k_retrieval` según latencia aceptable**: más candidatos = mejor recall, más latencia en reranking.
5. **Monitorear latencia por etapa**: alertar si la etapa de reranking supera el presupuesto (e.g., >200ms).
6. **Versionar índices**: guardar qué versión del modelo de embedding generó cada índice para poder regenerarlo.

### Errores comunes

1. **Reranking sin primera etapa**: evaluar el cross-encoder sobre todo el corpus es inviable (O(N) inferences).
2. **Normalizar scores antes de RRF**: RRF es robusto porque no requiere normalización → si normalizas, pierdes esa ventaja.
3. **FAISS bloqueante en async**: usar `run_in_executor` o thread pool para no bloquear el event loop de FastAPI/asyncio.
4. **Un solo índice para datos multi-tenant**: separar índices por tenant para evitar fugas de información y poder purgar datos.
5. **Ignorar el idioma del modelo de reranking**: `ms-marco-MiniLM` es para inglés; usar modelos multilingüe para español.

### Parámetros según caso de uso

```python
# Alta calidad, latencia tolerada (~200ms)
config_hq = RetrievalConfig(top_k_retrieval=50, top_k_rerank=10)

# Baja latencia, calidad razonable (~50ms)
config_ll = RetrievalConfig(top_k_retrieval=10, top_k_rerank=3, use_faiss=True)
```

## Conclusiones y Próximos Pasos

Implementamos un pipeline de recuperación de producción con cuatro componentes:

| Etapa | Componente | Aportación |
|---|---|---|
| 1a | BM25 | Recall en coincidencias exactas y léxicas |
| 1b | Dense + FAISS | Recall semántico escalable |
| 1c | RRF | Combinar rankings sin normalización de scores |
| 2 | Cross-encoder | Precisión alta en el top-K final |

**Pipeline recomendado**: `BM25 + Dense → RRF (top-20) → Cross-Encoder (top-5)`

### Próximos notebooks

- **`03_evaluation_metrics.ipynb`**: NDCG, análisis de fallos y LLM-as-judge para evaluar la calidad del sistema
- **`04_production_considerations.ipynb`**: Monitoreo, actualización incremental de índices y A/B testing

### Recursos adicionales

- [BEIR Benchmark](https://github.com/beir-cellar/beir) — comparativa estándar de métodos de retrieval
- [Sentence Transformers](https://www.sbert.net/) — modelos de embedding y cross-encoder
- [FAISS wiki](https://github.com/facebookresearch/faiss/wiki) — tipos de índices y cuándo usar cada uno
- ["Pretrained Transformers for Text Ranking"](https://arxiv.org/abs/2010.06467) — survey de modelos de reranking